# Week 02 - Time Series Forecasting with ARIMA

**DMML - Data Mining & Machine Learning**  
**Due:** End of Week 03  
**Estimated time:** 3-4 hours

This notebook is self-contained. You will work with the classic monthly airline passengers series and compare simple forecasting baselines against ARIMA-style models.


## What You Are Building

This week has five required functions:

1. `train_test_split_series(series, n_test)` - chronological splitting.
2. `run_adf_test(series, name)` - stationarity check.
3. `make_stationarity_table(series_dict)` - compare transformations.
4. `fit_arima_candidates(train, orders)` - compare ARIMA orders using AIC/BIC.
5. `evaluate_forecasts(test, forecasts)` - build a numeric benchmark for comparable forecasts.

The modelling lesson is more important than the syntax: time series must be split chronologically, simple baselines matter, and a low AIC model is not automatically the best forecast. At the end of the notebook can save these numbers in the shared benchmark table.

Each task ends with a **Tools for this task** list naming the library functions it needs, and what each one gives you. Working out the order to put them in is the exercise. The full reference lives in [`docs/libraries.html`](../../docs/libraries.html).


In [ ]:
# Imports - keep this cell stable
import warnings
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

RANDOM_STATE = 42
N_TEST = 24
SEASON_LENGTH = 12
ARIMA_CANDIDATES = [(0, 1, 1), (1, 1, 1), (2, 1, 1), (1, 1, 2), (2, 1, 2)]
AIR_PASSENGERS = [112, 118, 132, 129, 121, 135, 148, 148, 136, 119, 104, 118, 115, 126, 141, 135, 125, 149, 170, 170, 158, 133, 114, 140, 145, 150, 178, 163, 172, 178, 199, 199, 184, 162, 146, 166, 171, 180, 193, 181, 183, 218, 230, 242, 209, 191, 172, 194, 196, 196, 236, 235, 229, 243, 264, 272, 237, 211, 180, 201, 204, 188, 235, 227, 234, 264, 302, 293, 259, 229, 203, 229, 242, 233, 267, 269, 270, 315, 364, 347, 312, 274, 237, 278, 284, 277, 317, 313, 318, 374, 413, 405, 355, 306, 271, 306, 315, 301, 356, 348, 355, 422, 465, 467, 404, 347, 305, 336, 340, 318, 362, 348, 363, 435, 491, 505, 404, 359, 310, 337, 360, 342, 406, 396, 420, 472, 548, 559, 463, 407, 362, 405, 417, 391, 419, 461, 472, 535, 622, 606, 508, 461, 390, 432]


In [ ]:
# Build the monthly series directly so the notebook does not depend on network access.
idx = pd.date_range(start="1949-01-01", periods=len(AIR_PASSENGERS), freq="MS")
series = pd.Series(AIR_PASSENGERS, index=idx, name="passengers")
log_series = np.log(series)

print(series.head())
print(series.tail())


## Warm-Up: Plot the Series

The raw series has trend, seasonality, and increasing variance. The log transform should make the seasonal swings more stable across time.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharex=True)

series.plot(ax=axes[0], color="steelblue")
axes[0].set_title("Airline passengers")
axes[0].set_ylabel("Passengers (thousands)")

log_series.plot(ax=axes[1], color="darkorange")
axes[1].set_title("Log airline passengers")
axes[1].set_ylabel("log(passengers)")

for ax in axes:
    ax.axvline(series.index[-N_TEST], color="black", linestyle="--", alpha=0.7, label="test starts")
    ax.legend()

plt.tight_layout()
plt.show()


## Task 1 - Chronological Train/Test Split

Implement `train_test_split_series(series, n_test)`.

Rules:

- Return `(train, test)`.
- Keep the original datetime index.
- Do not shuffle.
- Raise a `ValueError` if `n_test <= 0` or `n_test >= len(series)`.

**Tools for this task** — reference: [pandas](../../docs/libraries.html#pandas) · [Python](../../docs/libraries.html#stdlib)

- `len(series)` — how many observations there are
- `s.iloc[:n] / s.iloc[-n:]` — positional slicing, which keeps the datetime index intact
- `raise ValueError('...')` — reject an invalid `n_test` before doing any work


In [ ]:
def train_test_split_series(series: pd.Series, n_test: int) -> tuple[pd.Series, pd.Series]:
    """Split a time series into chronological train and test portions."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 1
train, test = train_test_split_series(series, N_TEST)
log_train, log_test = train_test_split_series(log_series, N_TEST)

assert len(train) == len(series) - N_TEST
assert len(test) == N_TEST
assert train.index.max() < test.index.min(), "Train must come strictly before test"
assert train.iloc[-1] == series.iloc[-N_TEST - 1]
assert test.iloc[0] == series.iloc[-N_TEST]

try:
    train_test_split_series(series, len(series))
    raise AssertionError("Expected ValueError for invalid n_test")
except ValueError:
    pass

print("Task 1 passed")


## Guided Analysis: Decomposition

Use decomposition to inspect the trend and seasonal pattern. You do not need to wrap this in a function.


In [ ]:
# Multiplicative decomposition on the raw series
raw_decomp = seasonal_decompose(series, model="multiplicative", period=SEASON_LENGTH)
fig = raw_decomp.plot()
fig.set_size_inches(10, 7)
fig.suptitle("Raw series - multiplicative decomposition", y=1.02)
plt.tight_layout()
plt.show()

# Additive decomposition on the log series is often easier to reason about.
log_decomp = seasonal_decompose(log_series, model="additive", period=SEASON_LENGTH)
fig = log_decomp.plot()
fig.set_size_inches(10, 7)
fig.suptitle("Log series - additive decomposition", y=1.02)
plt.tight_layout()
plt.show()


## Task 2 - Stationarity Test

Implement `run_adf_test(series, name)`.

Return a dictionary with exactly these keys:

- `series`
- `adf_stat`
- `p_value`
- `critical_5pct`
- `stationary`

Drop missing values before running the test. Use `stationary = p_value < 0.05`.

**Tools for this task** — reference: [statsmodels](../../docs/libraries.html#statsmodels) · [pandas](../../docs/libraries.html#pandas)

- `s.dropna()` — the test cannot handle missing values
- `adfuller(series)` — returns a tuple: statistic, p-value, lags, nobs, then a dict of critical values
- `result[4]['5%']` — the 5% critical value lives in that dict


In [ ]:
def run_adf_test(series: pd.Series, name: str) -> dict:
    """Run the Augmented Dickey-Fuller test and return a compact result dict."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 2
adf_raw = run_adf_test(series, "Raw")
adf_diff = run_adf_test(log_series.diff().dropna(), "Log first difference")

expected_keys = {"series", "adf_stat", "p_value", "critical_5pct", "stationary"}
assert set(adf_raw.keys()) == expected_keys
assert isinstance(adf_raw["stationary"], (bool, np.bool_))
assert 0 <= adf_raw["p_value"] <= 1
assert adf_raw["series"] == "Raw"
assert adf_raw["p_value"] > 0.05, "The raw AirPassengers series should not look stationary"

print("Task 2 passed")
adf_raw


## Task 3 - Stationarity Comparison Table

Implement `make_stationarity_table(series_dict)`.

Input: a dictionary mapping display names to series.  
Output: a dataframe with columns:

- `series`
- `adf_stat`
- `p_value`
- `critical_5pct`
- `stationary`

Use your `run_adf_test` function internally.

**Tools for this task** — reference: [pandas](../../docs/libraries.html#pandas) · [Python](../../docs/libraries.html#stdlib)

- `d.items()` — iterate the name/series pairs
- `run_adf_test(...)` — your own Task 2 function — call it, do not reimplement it
- `pd.DataFrame(rows)` — one dict per row becomes the table


In [ ]:
def make_stationarity_table(series_dict: dict[str, pd.Series]) -> pd.DataFrame:
    """Run ADF tests on several series and return a comparison table."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 3
log_d1 = log_series.diff().dropna()
log_d12 = log_series.diff(SEASON_LENGTH).dropna()
log_d1_d12 = log_series.diff().diff(SEASON_LENGTH).dropna()

stationarity_table = make_stationarity_table({
    "Raw": series,
    "Log": log_series,
    "Log D1": log_d1,
    "Log D12": log_d12,
    "Log D1 D12": log_d1_d12,
})

assert isinstance(stationarity_table, pd.DataFrame)
assert list(stationarity_table.columns) == ["series", "adf_stat", "p_value", "critical_5pct", "stationary"]
assert len(stationarity_table) == 5
assert stationarity_table.loc[stationarity_table["series"] == "Raw", "stationary"].iloc[0] == False

print("Task 3 passed")
stationarity_table


## Guided Analysis: ACF and PACF

Inspect the differenced log series. Use the plots to choose a small set of ARIMA candidates rather than trying every possible order.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(log_d1.dropna(), lags=40, ax=axes[0])
axes[0].set_title("ACF: log first difference")
plot_pacf(log_d1.dropna(), lags=40, ax=axes[1], method="ywm")
axes[1].set_title("PACF: log first difference")
plt.tight_layout()
plt.show()

print("Candidate non-seasonal ARIMA orders:", ARIMA_CANDIDATES)


## Task 4 - Fit ARIMA Candidates

Implement `fit_arima_candidates(train, orders)`.

Fit each order on the log-transformed training series. Return a dataframe with columns:

- `order`
- `aic`
- `bic`
- `converged`
- `fit_time_sec`

Sort by `aic` ascending. If one candidate fails, include a row with `NaN` scores and `converged = False`.

**Tools for this task** — reference: [statsmodels](../../docs/libraries.html#statsmodels) · [NumPy](../../docs/libraries.html#numpy) · [pandas](../../docs/libraries.html#pandas) · [Python](../../docs/libraries.html#stdlib)

- `np.log(series)` — the log transform the task asks for
- `ARIMA(series, order=(p, d, q))` — the specification; `.fit()` gives the fitted results object
- `res.aic / res.bic` — information criteria, read off the results object
- `perf_counter()` — call it before and after the fit and take the difference
- `try: ... except Exception:` — record a failed order as NaN instead of crashing the notebook
- `df.sort_values('aic')` — ascending, so the best candidate is first


In [ ]:
def fit_arima_candidates(train: pd.Series, orders: list[tuple[int, int, int]]) -> pd.DataFrame:
    """Fit ARIMA candidates and return an AIC/BIC comparison table."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Self-check: Task 4
arima_candidates = fit_arima_candidates(log_train, ARIMA_CANDIDATES)

assert isinstance(arima_candidates, pd.DataFrame)
assert list(arima_candidates.columns) == ["order", "aic", "bic", "converged", "fit_time_sec"]
assert len(arima_candidates) == len(ARIMA_CANDIDATES)
assert arima_candidates["aic"].notna().any(), "At least one ARIMA candidate should fit"
assert arima_candidates["aic"].dropna().is_monotonic_increasing, "Sort by AIC ascending"

print("Task 4 passed")
arima_candidates


## Task 5 - Forecast Scoreboard

Implement `evaluate_forecasts(test, forecasts)`.

Input:

- `test`: actual values on the original passenger scale.
- `forecasts`: dictionary mapping model names to forecast series on the original passenger scale.

Return a dataframe with columns:

- `model`
- `mae`
- `rmse`
- `mape`

Sort by `rmse` ascending.

**Tools for this task** — reference: [NumPy](../../docs/libraries.html#numpy) · [pandas](../../docs/libraries.html#pandas)

- `d.items()` — iterate model name and forecast series
- `s.reindex(other.index)` — line a forecast up with the test index before subtracting
- `df.sort_values('rmse')` — ascending, best first

Writing `error = actual - forecast`, the three metrics are:

- **MAE** — `np.abs(error).mean()`
- **RMSE** — `np.sqrt((error ** 2).mean())`
- **MAPE** — `np.abs(error / actual).mean() * 100`, reported as a percentage

MAPE divides by the actual value, so it is undefined wherever an actual is zero. Passenger counts are never zero here, but guard for it if you reuse the function.


In [ ]:
def evaluate_forecasts(test: pd.Series, forecasts: dict[str, pd.Series]) -> pd.DataFrame:
    """Evaluate several forecasts against the same test window."""
    # YOUR CODE HERE
    raise NotImplementedError


In [ ]:
# Build comparable forecasts on the original passenger scale.

# Baseline 1: repeat the last training value.
naive_forecast = pd.Series(train.iloc[-1], index=test.index, name="naive_last_value")

# Baseline 2: repeat values from the same month one year earlier.
seasonal_values = train.iloc[-SEASON_LENGTH:].to_numpy()
seasonal_forecast = pd.Series(np.tile(seasonal_values, int(np.ceil(len(test) / SEASON_LENGTH)))[:len(test)], index=test.index, name="seasonal_naive_12")

# ARIMA: choose the lowest-AIC converged candidate and forecast on the log scale.
best_order = arima_candidates.dropna(subset=["aic"]).iloc[0]["order"]
best_model = ARIMA(log_train, order=best_order).fit()
log_forecast = best_model.get_forecast(steps=len(log_test)).predicted_mean
arima_forecast = np.exp(log_forecast)
arima_forecast.name = f"ARIMA{best_order}"

forecast_table = evaluate_forecasts(test, {
    "naive_last_value": naive_forecast,
    "seasonal_naive_12": seasonal_forecast,
    f"ARIMA{best_order}": arima_forecast,
})

assert isinstance(forecast_table, pd.DataFrame)
assert list(forecast_table.columns) == ["model", "mae", "rmse", "mape"]
assert len(forecast_table) == 3
assert forecast_table["rmse"].is_monotonic_increasing, "Sort by RMSE ascending"
assert forecast_table[["mae", "rmse", "mape"]].notna().all().all()

print("Task 5 passed")
forecast_table


## Forecast Plot

Use the table above to compare models numerically, then inspect the forecast shape visually.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
train.plot(ax=ax, label="train", color="steelblue")
test.plot(ax=ax, label="test", color="black")
naive_forecast.plot(ax=ax, label="naive", linestyle="--")
seasonal_forecast.plot(ax=ax, label="seasonal naive", linestyle="--")
arima_forecast.plot(ax=ax, label=f"ARIMA{best_order}", linestyle="--")
ax.axvline(test.index[0], color="black", linestyle=":", alpha=0.7)
ax.set_title("Forecast comparison on held-out months")
ax.set_ylabel("Passengers (thousands)")
ax.legend()
plt.tight_layout()
plt.show()


## Save to the Course Benchmark

Every week writes its scores into one shared table, `common/benchmark_results.csv`.
Run this cell once your numbers are final, then open `../common/benchmark.ipynb`
to see this week next to everything you have run so far.

These are the first rows of the table. Forecasting is scored with error metrics, where lower is better - the opposite direction from every accuracy you will record later.


In [ ]:
# Save this week's results to the shared, course-long benchmark table.
import sys
sys.path.append("../common")
from dmml_benchmark import save_results

save_results(
    forecast_table,
    week="W02",
    dataset="AirPassengers",
    task_type="forecasting",
    target="passengers",
    split="last_24_months",
    metrics=["mae", "rmse", "mape"],
    notes="monthly data; chronological holdout",
)


## Challenge Tracks Optional

Choose zero, one, or more.

### Track A - Seasonal ARIMA
Try a SARIMA model with a seasonal order such as `(1, 1, 1, 12)` and add it to the shared benchmark with a second `save_results` call.

### Track B - Residual Diagnostics
Plot residuals from the best ARIMA model and run a Ljung-Box test. Are the residuals close to white noise?

### Track C - Robustness
Change the test window to 12 months and then 36 months. Does the ranking of models change?


In [ ]:
# Optional challenge workspace
# Your code here
